In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src import fdm_schemes
import matplotlib as mpl
import matplotlib.animation as animation

# Choose a base font size
base_fs = 14

# Global update (applies for the rest of this Python session)
mpl.rcParams.update({
    'font.size': base_fs,                   # default text size
    'axes.titlesize': base_fs * 1.1,       # axes title
    'axes.labelsize': base_fs,              # x/y labels
    'xtick.labelsize': base_fs * 0.9,       # x tick labels
    'ytick.labelsize': base_fs * 0.9,       # y tick labels
    'legend.fontsize': base_fs * 0.9,       # legend text
    'legend.title_fontsize': base_fs * 0.9, # legend title
    'figure.titlesize': base_fs * 1.3,      # figure suptitle
    'figure.figsize': (8, 4.5),             # optional default figure size
})

In [ ]:

L = 1.0
N = 1000
dx = L / (N-1)
dt = 0.001
T = 2.0
c = 1.0

u0s = [np.sin(2 * np.pi * np.linspace(0, L, N)),
       np.sin(5 * np.pi * np.linspace(0, L, N)),
       np.sin(5 * np.pi * np.linspace(0, L, N))]

u0s[-1][:int(np.ceil(1/5*(N-1)))] = 0
u0s[-1][int(np.floor(2/5*(N-1))+1):] = 0

results = []
for u0 in u0s:
    res = fdm_schemes.wave_equation_1d(u0, c, dx, dt, T)
    results.append(res)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12
                                       , 4), constrained_layout=True)
initial_conditions = ['sin(2πx)', 'sin(5πx)',     'sin(5πx) with \n zero outside [0.2, 0.4]']

# python
# create a single colorbar for all three panels by using a shared vmin/vmax
vmin = min(res.min() for res in results)
vmax = max(res.max() for res in results)


for i, res in enumerate(results):
    im = axs[i].imshow(res, aspect='auto', extent=(0, L, T, 0), vmin=vmin, vmax=vmax)

    axs[i].set_title(f'Initial condition: \n' + fr'$u_0$ = {initial_conditions[i]}')
    axs[i].set_xlabel('x')
    axs[i].set_ylabel('Time (s)')

fig.colorbar(im, ax=axs, location='right', label='Amplitude (u)')
plt.suptitle('1D wave equation solution over time for different initial conditions', y=1.06)
plt.show()

In [ ]:

periods = np.array([1, 1/2.5, 2])
step_sizes = 100 * periods

fig, axs = plt.subplots(3, 1, figsize=(8, 7), constrained_layout=True)

stagger_offsets = [0.05, 0.00, -0.05]
pad = -0.9

x = np.linspace(0, L, N)

for i, res in enumerate(results):
    step_size = max(1, int(step_sizes[i]))
    period = periods[i]
    idxs = np.arange(0, int(period / (2 * dt)) + 1, step_size)   # time indices to plot
    times = idxs * dt

    cmap = plt.get_cmap('viridis')
    norm = mpl.colors.Normalize(vmin=times.min(), vmax=times.max())
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array(times)

    # plot each time-slice with color from the colormap
    for j, idx in enumerate(idxs):
        axs[i].plot(x, res[idx, :], color=cmap(norm(times[j])), lw=1)

    axs[i].set_xlabel('x')
    axs[i].set_ylabel('Amplitude (u)')
    axs[i].set_title('')   # keep per-row title empty (use side label)

    # small colorbar per subplot showing the time mapping
    cbar = fig.colorbar(sm, ax=axs[i], orientation='vertical', pad=0.02)
    cbar.set_label('time (s)')

# place staggered vertical side labels to the left of each axis
for i, ax in enumerate(axs):
    pos = ax.get_position()
    x = pos.x0 - pad
    y = pos.y0 + pos.height / 2 + stagger_offsets[i]
    label = fr'$u_0$ = {initial_conditions[i]}'
    fig.text(x, y, label, rotation='vertical', va='center', ha='center',
             fontsize=base_fs * 0.95, bbox=dict(facecolor='white', alpha=0.0, edgecolor='none'))

plt.suptitle('1D wave snapshots for different initial conditions', y=1.04)
plt.show()

In [ ]:
# Animation helper: create and save an animation of the 1D wave
# The user can set `out_filename` below before running this cell.

def save_wave_animation(u_time_space, x, dt, out_filename='wave_animation.gif', frame_step=None, dpi=150):
    """
    Create and save an animation of a 1D wave solution.

    Parameters
    ----------
    u_time_space : ndarray
        Array of shape (nt, nx) containing solution at each time step.
    x : ndarray
        Spatial coordinates of length nx.
    dt : float
        Time step used between frames (seconds).
    out_filename : str
        Output filename (recommended extension: .gif or .mp4).
    frame_step : int or None
        Save every `frame_step`-th frame. If None, it will be chosen so the
        output has at most ~200 frames.
    dpi : int
        Resolution for the saved animation.
    """
    nt, nx = u_time_space.shape

    if frame_step is None:
        frame_step = max(1, int(nt / 200))

    frames = list(range(0, nt, frame_step))
    interval_ms = dt * 1000 * frame_step

    fig, ax = plt.subplots()
    line, = ax.plot(x, u_time_space[0])
    ax.set_xlim(x.min(), x.max())
    y_margin = 0.1 * (u_time_space.max() - u_time_space.min())
    if y_margin == 0:
        y_margin = 1.0
    ax.set_ylim(u_time_space.min() - y_margin, u_time_space.max() + y_margin)
    ax.set_xlabel('x')
    ax.set_ylabel('u')
    ax.set_title('1D wave')

    def update(i):
        line.set_ydata(u_time_space[i])
        ax.set_title(f'Time = {i*dt:.3f} s')
        return (line,)

    anim = animation.FuncAnimation(fig, update, frames=frames, interval=interval_ms, blit=True)

    # Try to use PillowWriter (no external ffmpeg required) when saving GIFs
    try:
        from matplotlib.animation import PillowWriter
        if out_filename.lower().endswith('.gif'):
            writer = PillowWriter(fps=max(1, int(1000/interval_ms)))
            anim.save(out_filename, writer=writer, dpi=dpi)
        else:
            # For mp4, fall back to ffmpeg if available
            anim.save(out_filename, dpi=dpi)
    except Exception:
        # Fallback: try default save (may require ffmpeg)
        anim.save(out_filename, dpi=dpi)

    plt.close(fig)
    print(f"Saved animation to: {out_filename}")


# User-configurable output filename (change this before running)
out_filename = 'assignment1_2pi.gif'
# Build x vector and call the saver
x = np.linspace(0, L, N)
save_wave_animation(res, x, dt, out_filename=out_filename)
